In [ ]:
#Problem 1] Simple Forward propagation implementation of RNN

In [1]:
import numpy as np

# --- Helper Functions (Activations) ---
def tanh(X):
    """Tanh activation function."""
    return np.tanh(X)

# --- Helper Functions (Initializers) ---
# Simple placeholder initializers for weights and bias
def simple_initializer(shape):
    """Initialize weights using a standard normal distribution."""
    return np.random.randn(*shape) * 0.01

def zero_initializer(shape):
    """Initialize bias terms to zeros."""
    return np.zeros(shape)

# ====================================================================
# Problem 1: SimpleRNN Layer Implementation
# ====================================================================

class SimpleRNN:
    """
    Implements the forward propagation for a Simple Recurrent Layer (RNN).

    The formula at time t is:
    a_t = x_t @ W_x + h_{t-1} @ W_h + B
    h_t = tanh(a_t)
    """

    def __init__(self, n_nodes, Wx_initializer=simple_initializer, Wh_initializer=simple_initializer, B_initializer=zero_initializer, activation=tanh):
        """
        Initializes the SimpleRNN layer weights and bias.

        Parameters
        ----------
        n_nodes : int
            The number of nodes (units) in the RNN layer (latent dimension).
        Wx_initializer : function
            Initializer for the input weights (W_x).
        Wh_initializer : function
            Initializer for the recurrent weights (W_h).
        B_initializer : function
            Initializer for the bias term (B).
        activation : function
            The activation function for the hidden state (e.g., tanh).
        """
        self.n_nodes = n_nodes
        self.Wx_initializer = Wx_initializer
        self.Wh_initializer = Wh_initializer
        self.B_initializer = B_initializer
        self.activation = activation
        
        # Weights and bias are initialized in the first forward pass 
        # when n_features is known.
        self.W_x = None
        self.W_h = None
        self.B = None
        
        # Store all hidden states during forward propagation
        self.h_sequence = None
        # Store all activation states (a_t) for potential backprop
        self.a_sequence = None

    def initialize_weights(self, n_features):
        """Initializes weights W_x, W_h, and bias B once the input features are known."""
        if self.W_x is None:
            # W_x shape: (n_features, n_nodes)
            self.W_x = self.Wx_initializer((n_features, self.n_nodes))
            # W_h shape: (n_nodes, n_nodes)
            self.W_h = self.Wh_initializer((self.n_nodes, self.n_nodes))
            # B shape: (n_nodes,)
            self.B = self.B_initializer((self.n_nodes,))
            print(f"RNN Weights Initialized: Wx={self.W_x.shape}, Wh={self.W_h.shape}, B={self.B.shape}")

    def forward(self, X, H_prev=None):
        """
        Performs the forward propagation of the Simple RNN layer.

        Parameters
        ----------
        X : ndarray, shape (batch_size, n_sequences, n_features)
            The input data sequence.
        H_prev : ndarray, shape (batch_size, n_nodes), optional
            The initial hidden state (h_0). Defaults to zeros if None.

        Returns
        -------
        H_sequence : ndarray, shape (batch_size, n_sequences, n_nodes)
            The sequence of all hidden states [h_1, h_2, ..., h_T].
        H_last : ndarray, shape (batch_size, n_nodes)
            The final hidden state (h_T).
        """
        batch_size, n_sequences, n_features = X.shape
        
        # Initialize weights if this is the first time running forward
        self.initialize_weights(n_features)
        
        # If H_prev (h_0) is not provided, initialize it to zeros
        if H_prev is None:
            H_prev = np.zeros((batch_size, self.n_nodes))
        
        # Initialize arrays to store the sequence of states
        H_sequence = np.zeros((batch_size, n_sequences, self.n_nodes))
        A_sequence = np.zeros((batch_size, n_sequences, self.n_nodes))
        
        # --- Recurrent Forward Loop ---
        for t in range(n_sequences):
            # 1. Slice the input for the current time step t
            X_t = X[:, t, :]  # Shape: (batch_size, n_features)
            
            # 2. State before activation (a_t)
            # a_t = x_t @ W_x + h_{t-1} @ W_h + B
            # x_t @ W_x: (batch_size, n_nodes)
            # h_{t-1} @ W_h: (batch_size, n_nodes)
            # B: (n_nodes,) broadcasted to (batch_size, n_nodes)
            
            A_t = X_t @ self.W_x + H_prev @ self.W_h + self.B
            
            # 3. Hidden state (h_t)
            H_t = self.activation(A_t) # Shape: (batch_size, n_nodes)
            
            # 4. Store the states for the current time step
            H_sequence[:, t, :] = H_t
            A_sequence[:, t, :] = A_t
            
            # 5. Update the state for the next time step: h_{t} becomes h_{t-1}
            H_prev = H_t
            
        # Store for potential backpropagation
        self.h_sequence = H_sequence
        self.a_sequence = A_sequence

        # Return the full sequence and the final state
        return H_sequence, H_prev

# ====================================================================
# Classifier Structure (Demonstrating Usage)
# ====================================================================

class ScratchSimpleRNNClassifier:
    """
    A minimal classifier structure using the SimpleRNN layer.
    (Forward pass only for demonstration)
    """
    def __init__(self, n_nodes, n_output):
        # 1. RNN Layer
        self.rnn = SimpleRNN(n_nodes=n_nodes)
        
        # 2. Output Layer (Fully Connected)
        # Note: In a full implementation, this should be a proper FC class
        self.W_out = simple_initializer((n_nodes, n_output))
        self.B_out = zero_initializer((n_output,))
        
    def softmax(self, X):
        """Softmax function for final classification."""
        exp_X = np.exp(X - np.max(X, axis=1, keepdims=True))
        return exp_X / np.sum(exp_X, axis=1, keepdims=True)

    def forward(self, X):
        """
        Performs the forward pass for the entire classifier.
        Uses only the last hidden state (h_T) for classification.
        """
        # X: (batch_size, n_sequences, n_features)
        
        # 1. RNN Forward Pass
        # H_sequence: (batch_size, n_sequences, n_nodes)
        # H_last: (batch_size, n_nodes)
        H_sequence, H_last = self.rnn.forward(X)
        
        # 2. Output Layer (Classification uses only the last state H_last)
        # Z_out = H_last @ W_out + B_out
        Z_out = H_last @ self.W_out + self.B_out # (batch_size, n_output)
        
        # 3. Softmax Activation
        Y_pred = self.softmax(Z_out)
        
        return Y_pred, H_sequence

# --- Example Usage ---
if __name__ == '__main__':
    # Define parameters
    batch_size = 4
    n_sequences = 10
    n_features = 5
    n_nodes = 32
    n_output = 2  # Binary Classification (0 or 1)
    
    # 1. Create dummy input data
    # (batch_size, n_sequences, n_features)
    X_dummy = np.random.randn(batch_size, n_sequences, n_features)
    
    print("--- Running SimpleRNN Layer Forward Pass ---")
    
    # Initialize the SimpleRNN layer
    rnn_layer = SimpleRNN(n_nodes=n_nodes)
    
    # Run the forward pass (h_0 is initialized to zeros automatically)
    H_sequence, H_last = rnn_layer.forward(X_dummy)
    
    print(f"Input Shape (X): {X_dummy.shape}")
    print(f"Output Sequence Shape (H_sequence): {H_sequence.shape}")
    print(f"Final State Shape (H_last): {H_last.shape}")
    
    # Verification of shapes
    assert H_sequence.shape == (batch_size, n_sequences, n_nodes)
    assert H_last.shape == (batch_size, n_nodes)
    print("\nShape verification successful.")

    # --- Running Classifier Forward Pass ---
    print("\n--- Running ScratchSimpleRNNClassifier Forward Pass ---")

    # Initialize the Classifier
    rnn_classifier = ScratchSimpleRNNClassifier(n_nodes=n_nodes, n_output=n_output)
    
    # Run the full forward pass
    Y_pred, _ = rnn_classifier.forward(X_dummy)
    
    print(f"Output Prediction Shape (Y_pred): {Y_pred.shape}")
    print(f"Sum of probabilities for one sample: {Y_pred[0].sum():.6f}")

    assert Y_pred.shape == (batch_size, n_output)
    print("Classifier shape verification successful.")


--- Running SimpleRNN Layer Forward Pass ---
RNN Weights Initialized: Wx=(5, 32), Wh=(32, 32), B=(32,)
Input Shape (X): (4, 10, 5)
Output Sequence Shape (H_sequence): (4, 10, 32)
Final State Shape (H_last): (4, 32)

Shape verification successful.

--- Running ScratchSimpleRNNClassifier Forward Pass ---
RNN Weights Initialized: Wx=(5, 32), Wh=(32, 32), B=(32,)
Output Prediction Shape (Y_pred): (4, 2)
Sum of probabilities for one sample: 1.000000
Classifier shape verification successful.


In [ ]:
#[Problem 2] Experiment of forward propagation with small sequence

In [2]:
# --- Example Usage ---
if __name__ == '__main__':
    
    # ====================================================================
    # Problem 2: Experiment with Small Fixed Arrays (Verification Test)
    # ====================================================================

    # --- Fixed Data and Initializers for Verification ---
    X_fixed = np.array([[[1, 2], [2, 3], [3, 4]]]) / 100  # (1, 3, 2)
    Wx_fixed = np.array([[1, 3, 5, 7], [3, 5, 7, 8]]) / 100  # (2, 4)
    Wh_fixed = (
        np.array([[1, 3, 5, 7], [2, 4, 6, 8], [3, 5, 7, 8], [4, 6, 8, 10]]) / 100
    )  # (4, 4)
    B_fixed = np.array([1, 1, 1, 1])  # (4,)

    def fixed_Wx_initializer(shape):
        # Ignores the passed shape and returns the fixed Wx
        return Wx_fixed

    def fixed_Wh_initializer(shape):
        # Ignores the passed shape and returns the fixed Wh
        return Wh_fixed

    def fixed_B_initializer(shape):
        # Ignores the passed shape and returns the fixed B
        return B_fixed

    # Expected final hidden state h_3 based on manual calculation
    H_expected_last = np.array(
        [[0.79494228, 0.81839002, 0.83939649, 0.85584174]]
    )

    print("--- Running Problem 2 Verification Test (Fixed Arrays) ---")
    n_nodes_fixed = Wx_fixed.shape[1] # 4

    # Initialize RNN layer using fixed initializers
    rnn_test = SimpleRNN(
        n_nodes=n_nodes_fixed,
        Wx_initializer=fixed_Wx_initializer,
        Wh_initializer=fixed_Wh_initializer,
        B_initializer=fixed_B_initializer
    )

    # Run the forward pass (h_0 is np.zeros)
    _, H_actual_last = rnn_test.forward(X_fixed)

    print(f"Input X_fixed shape: {X_fixed.shape}")
    print(f"Actual Final State (H_last) shape: {H_actual_last.shape}")

    # Check for close numerical equality (using tolerance for floating point numbers)
    is_close = np.allclose(H_actual_last, H_expected_last, rtol=1e-06, atol=1e-06)

    print(f"Verification Successful? {is_close}")
    if is_close:
        print("RESULT: The calculated final hidden state matches the expected value.")
    else:
        print("ERROR: The calculated final hidden state DOES NOT match the expected value.")
        
    print("----------------------------------------------------------\n")


    # --- Example Usage (Original Dummy Test) ---
    # Define parameters
    batch_size = 4
    n_sequences = 10
    n_features = 5
    n_nodes = 32
    n_output = 2  # Binary Classification (0 or 1)
    
    # 1. Create dummy input data
    # (batch_size, n_sequences, n_features)
    X_dummy = np.random.randn(batch_size, n_sequences, n_features)
    
    print("--- Running SimpleRNN Layer Forward Pass ---")
    
    # Initialize the SimpleRNN layer
    rnn_layer = SimpleRNN(n_nodes=n_nodes)
    
    # Run the forward pass (h_0 is initialized to zeros automatically)
    H_sequence, H_last = rnn_layer.forward(X_dummy)
    
    print(f"Input Shape (X): {X_dummy.shape}")
    print(f"Output Sequence Shape (H_sequence): {H_sequence.shape}")
    print(f"Final State Shape (H_last): {H_last.shape}")
    
    # Verification of shapes
    assert H_sequence.shape == (batch_size, n_sequences, n_nodes)
    assert H_last.shape == (batch_size, n_nodes)
    print("\nShape verification successful.")

    # --- Running Classifier Forward Pass ---
    print("\n--- Running ScratchSimpleRNNClassifier Forward Pass ---")

    # Initialize the Classifier
    rnn_classifier = ScratchSimpleRNNClassifier(n_nodes=n_nodes, n_output=n_output)
    
    # Run the full forward pass
    Y_pred, _ = rnn_classifier.forward(X_dummy)
    
    print(f"Output Prediction Shape (Y_pred): {Y_pred.shape}")
    print(f"Sum of probabilities for one sample: {Y_pred[0].sum():.6f}")

    assert Y_pred.shape == (batch_size, n_output)
    print("Classifier shape verification successful.")


--- Running Problem 2 Verification Test (Fixed Arrays) ---
RNN Weights Initialized: Wx=(2, 4), Wh=(4, 4), B=(4,)
Input X_fixed shape: (1, 3, 2)
Actual Final State (H_last) shape: (1, 4)
Verification Successful? True
RESULT: The calculated final hidden state matches the expected value.
----------------------------------------------------------

--- Running SimpleRNN Layer Forward Pass ---
RNN Weights Initialized: Wx=(5, 32), Wh=(32, 32), B=(32,)
Input Shape (X): (4, 10, 5)
Output Sequence Shape (H_sequence): (4, 10, 32)
Final State Shape (H_last): (4, 32)

Shape verification successful.

--- Running ScratchSimpleRNNClassifier Forward Pass ---
RNN Weights Initialized: Wx=(5, 32), Wh=(32, 32), B=(32,)
Output Prediction Shape (Y_pred): (4, 2)
Sum of probabilities for one sample: 1.000000
Classifier shape verification successful.


In [ ]:
#[Problem 3] (Advance assignment) Implementation of backpropagation

In [5]:
# --- Backpropagation Demonstration Setup ---
alpha = 0.01
batch_size = 4
n_sequences = 10
n_features = 5
n_nodes = 32
n_output = 2

print("--- Running Problem 3 Backpropagation Demonstration ---")

# 1. Setup Data and Classifier
X_dummy = np.random.randn(batch_size, n_sequences, n_features)
# Create one-hot labels for classification
Y_true_labels = np.random.randint(0, n_output, size=batch_size)
Y_true = np.zeros((batch_size, n_output))
Y_true[np.arange(batch_size), Y_true_labels] = 1

rnn_classifier = ScratchSimpleRNNClassifier(n_nodes=n_nodes, n_output=n_output)

# Store initial weights for comparison
W_out_initial = rnn_classifier.W_out.copy()
# NOTE: Wx is initialized inside forward() if it hasn't been run yet
Y_pred, _ = rnn_classifier.forward(X_dummy) 
Wx_initial = rnn_classifier.rnn.W_x.copy()

print(f"Initial Wx sum before backprop: {np.sum(Wx_initial):.4f}")
print(f"Initial W_out sum before backprop: {np.sum(W_out_initial):.4f}")

# 2. Backward Pass
# This updates Wx, Wh, B, W_out, and B_out
grad_X = rnn_classifier.backward(X_dummy, Y_true, alpha=alpha)

# 3. Verification
W_out_final = rnn_classifier.W_out
Wx_final = rnn_classifier.rnn.W_x

print(f"Final Wx sum after backprop: {np.sum(Wx_final):.4f}")
print(f"Final W_out sum after backprop: {np.sum(W_out_final):.4f}")
print(f"Gradient w.r.t Input (dL/dX) shape: {grad_X.shape}")

# Check if weights have been updated
assert not np.allclose(Wx_initial, Wx_final)
assert not np.allclose(W_out_initial, W_out_final)
print("\nSuccess: Weight updates verified after backpropagation.")
print("----------------------------------------------------------")

--- Running Problem 3 Backpropagation Demonstration ---
RNN Weights Initialized: Wx=(5, 32), Wh=(32, 32), B=(32,)
Initial Wx sum before backprop: 0.0772
Initial W_out sum before backprop: 0.1106


AttributeError: 'ScratchSimpleRNNClassifier' object has no attribute 'backward'